## Let's try gemma-4-E4B-it
it - instruction tuned
E4B - 4 B of effective parameters

https://huggingface.co/blog/gemma4#overview-of-capabilities-and-architecture



In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "3" #without it gptq will try to start on all gpus
from transformers import AutoModelForCausalLM
import torch
from dotenv import load_dotenv
load_dotenv("/glazkov-dev/.env") #for hf token and cache directory
# from transformers import BitsAndBytesConfig
# from gptqmodel import GPTQModel

True

In [2]:
MODEL = "google/gemma-4-E4B-it" 
# quantization_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
#         bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
#         # Normalized float 4-bit (optimal for LLMs)
#         bnb_4bit_quant_type="nf4",
#         llm_int8_skip_modules=[
#             "lm_head",
#             "mlp.gate", #because DeepSeekV2ForCausalLM has a f.linear(self.gate.weight...) instead of self.gate()
#             #it couldn't be substituded with Linear4bit
#         ],
#     )

In [3]:
import torch
print("2.10.0+cu128 we need on server")
torch.__version__

2.10.0+cu128 we need on server


'2.10.0+cu128'

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    #quantization_config=quantization_config,
    #dtype=torch.bfloat16,
    device_map="cuda:3",
    # cache_dir="/glazkov-dev/.cache",
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [5]:
import torch

print("torch:", torch.__version__)
print("compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
compiled CUDA: 12.8
CUDA available: True
device: NVIDIA A100-PCIE-40GB


In [6]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [7]:
import transformer_lens
print(transformer_lens.__file__)

/glazkov-dev/TransformerLens/transformer_lens/__init__.py


In [8]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [9]:
from utils.mmlu_batch_generator import MMLUBatchGenerator
SELECTED_TASKS = ["anatomy",
            # "conceptual_physics",
            # "human_sexuality",
            "machine_learning",
            "management",
            # "marketing",
            # "nutrition",
             "philosophy",
            # "us_foreign_policy",
              "world_religions"]
gen = MMLUBatchGenerator(subjects=SELECTED_TASKS, split='validation', batch_size=16, include_metadata=False)

Initialized MMLU batch generator with 5 subjects
Split: validation, Batch size: 16


In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL,
                                                device_map='cuda:3',
                                                )

for batch in gen:
    inputs = tokenizer(batch, return_tensors="pt",
            padding=True,  # Pad to longest in batch
            truncation=True,
            max_length=512,  # Adjust based on your needs
            add_special_tokens=True
    )  
    inputs = {k: v.to(bridge.device) for k, v in inputs.items()}
    print(inputs.keys())
    bridge_outputs, cache = bridge.run_with_cache(inputs['input_ids'], attention_mask=inputs['attention_mask'])
    print(bridge_outputs)  # Example: (batch_size, sequence_length, vocab_size)
    break  # Remove this break to process all batches

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📚 Loading subject 1/5: anatomy
  ✓ Created batch with 14 questions
dict_keys(['input_ids', 'attention_mask'])
tensor([[[ -8.0625,   1.8203,  -3.8906,  ...,  -8.0625,  -8.1250,  -8.0625],
         [ -8.0625,   1.8203,  -3.8906,  ...,  -8.0625,  -8.1250,  -8.0625],
         [ -8.0625,   1.8203,  -3.8906,  ...,  -8.0625,  -8.1250,  -8.0625],
         ...,
         [-17.2500,  -5.4688, -12.6250,  ..., -17.0000, -17.0000, -17.1250],
         [-16.1250,  -2.6562, -10.1875,  ..., -16.1250, -16.1250, -16.1250],
         [-13.6875,   1.7734,  -7.6250,  ..., -13.7500, -13.6875, -13.6875]],

        [[ -8.0625,   1.8203,  -3.8906,  ...,  -8.0625,  -8.1250,  -8.0625],
         [ -8.0625,   1.8203,  -3.8906,  ...,  -8.0625,  -8.1250,  -8.0625],
         [ -8.0625,   1.8203,  -3.8906,  ...,  -8.0625,  -8.1250,  -8.0625],
         ...,
         [-14.4375,   0.1689,  -9.4375,  ..., -14.5000, -14.5000, -14.4375],
         [-16.1250,   1.0234,  -8.0000,  ..., -16.1250, -16.1250, -16.1250],
         [-1

In [11]:
bridge_outputs.shape

torch.Size([14, 30, 262144])

Must be float16 everywhere

In [21]:
bridge

TransformerBridge(
  (vision_encoder): GeneralizedComponent(
    (hook_in): HookPoint(name='vision_encoder.hook_in')
    (hook_out): HookPoint(name='vision_encoder.hook_out')
    (_original_component): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              

In [26]:
bridge.model.language_model.embed_tokens

EmbeddingBridge(
  (hook_in): HookPoint(name='embed.hook_in')
  (hook_out): HookPoint(name='embed.hook_out')
  (_original_component): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
)

In [32]:


print("embedding:", bridge.model.language_model.embed_tokens.weight.dtype)
print(
    "norm:",
    bridge.model.language_model.layers[0].input_layernorm.weight.dtype,
)
print(
    "q scales:",
    bridge.model.language_model.layers[0].self_attn.q.weight.dtype,
)

embedding: torch.bfloat16
norm: torch.bfloat16
q scales: torch.bfloat16


In [38]:
attn = bridge.model.language_model.layers[0].self_attn

print(type(attn))
print(type(attn.q_proj))

<class 'transformer_lens.model_bridge.generalized_components.base.GeneralizedComponent'>
<class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>


In [39]:
print(type(bridge.model.language_model.layers[0].self_attn))
print(type(bridge.model.language_model.layers[0].self_attn.q))
print(type(bridge.model.language_model.layers[0].self_attn.q.original_component))

<class 'transformer_lens.model_bridge.generalized_components.base.GeneralizedComponent'>
<class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>
<class 'torch.nn.modules.linear.Linear'>


In [40]:
import torch
from collections import defaultdict

hook_handles = []
call_counts = defaultdict(int)

def tensor_info(value):
    if isinstance(value, torch.Tensor):
        return {
            "shape": tuple(value.shape),
            "dtype": str(value.dtype),
            "device": str(value.device),
        }

    if isinstance(value, (tuple, list)):
        return [
            tensor_info(item)
            for item in value
            if isinstance(item, (torch.Tensor, tuple, list, dict))
        ]

    if isinstance(value, dict):
        return {
            key: tensor_info(item)
            for key, item in value.items()
            if isinstance(item, (torch.Tensor, tuple, list, dict))
        }

    return type(value).__name__


def register_debug_hooks(
    root,
    *,
    max_calls_per_module=1,
    leaf_only=True,
    name_filter=None,
):
    """
    root: hf_model, bridge или отдельный блок.
    max_calls_per_module: сколько вызовов каждого модуля печатать.
    leaf_only: печатать только модули без дочерних компонентов.
    name_filter: функция (name, module) -> bool.
    """
    handles = []
    counts = defaultdict(int)

    for name, module in root.named_modules():
        if not name:
            continue

        if leaf_only and any(module.children()):
            continue

        if name_filter is not None and not name_filter(name, module):
            continue

        module_name = name
        module_type = type(module).__name__

        def pre_hook(mod, args, kwargs, *, _name=module_name, _type=module_type):
            if counts[(_name, "pre")] >= max_calls_per_module:
                return

            counts[(_name, "pre")] += 1

            print(f"\n→ {_name} [{_type}]")
            print("  args:", tensor_info(args))

            if kwargs:
                print("  kwargs:", tensor_info(kwargs))

        def post_hook(
            mod,
            args,
            kwargs,
            output,
            *,
            _name=module_name,
            _type=module_type,
        ):
            if counts[(_name, "post")] >= max_calls_per_module:
                return

            counts[(_name, "post")] += 1
            print(f"← {_name} [{_type}]")
            print("  output:", tensor_info(output))

        handles.append(
            module.register_forward_pre_hook(
                pre_hook,
                with_kwargs=True,
            )
        )

        handles.append(
            module.register_forward_hook(
                post_hook,
                with_kwargs=True,
            )
        )

    print(f"Registered {len(handles)} hooks")
    return handles

In [41]:
handles = register_debug_hooks(
    bridge,
    max_calls_per_module=1,
    leaf_only=True,
)

Registered 5566 hooks


In [46]:
type(bridge.model.language_model.layers[0].self_attn.q)

transformer_lens.model_bridge.generalized_components.linear.LinearBridge

In [42]:
with torch.inference_mode():
    outputs, cache = bridge.run_with_cache(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )


→ embed.hook_in [HookPoint]
  args: [{'shape': (14, 30), 'dtype': 'torch.int64', 'device': 'cuda:3'}]
← embed.hook_in [HookPoint]
  output: {'shape': (14, 30), 'dtype': 'torch.int64', 'device': 'cuda:3'}

→ embed._original_component [Gemma4TextScaledWordEmbedding]
  args: [{'shape': (14, 30), 'dtype': 'torch.int64', 'device': 'cuda:3'}]
← embed._original_component [Gemma4TextScaledWordEmbedding]
  output: {'shape': (14, 30, 2560), 'dtype': 'torch.bfloat16', 'device': 'cuda:3'}

→ embed.hook_out [HookPoint]
  args: [{'shape': (14, 30, 2560), 'dtype': 'torch.bfloat16', 'device': 'cuda:3'}]
← embed.hook_out [HookPoint]
  output: {'shape': (14, 30, 2560), 'dtype': 'torch.bfloat16', 'device': 'cuda:3'}

→ rotary_emb.hook_in [HookPoint]
  args: [{'shape': (14, 30, 2560), 'dtype': 'torch.bfloat16', 'device': 'cuda:3'}]
← rotary_emb.hook_in [HookPoint]
  output: {'shape': (14, 30, 2560), 'dtype': 'torch.bfloat16', 'device': 'cuda:3'}

→ rotary_emb._original_component [Gemma4TextRotaryEmbeddin